In [ ]:
# Install once if needed: %pip install torch torchvision matplotlib pandas tqdm
import copy, math, random, time
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| torch:', torch.__version__)

In [ ]:
def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@dataclass
class Config:
    dataset: str = 'CIFAR10'
    n_tasks: int = 5
    buffer_size: int = 200
    epochs: int = 1
    batch_size: int = 128
    minibatch_size: int = 128
    lr: float = 0.1
    momentum: float = 0.9
    weight_decay: float = 5e-4
    empty_probability: float = 0.5
    alpha: float = 1.0       # L_ide
    beta: float = 1.0        # L_rep-ice
    class_balance: bool = False
    num_workers: int = 2
    data_root: str = './data'

RUN_CONFIG = Config()
seed_everything(0)
print(asdict(RUN_CONFIG))

## 1. Class-incremental data stream

CIFAR-10 uses 5 tasks with 2 classes per task; CIFAR-100 uses 10 tasks with 10 classes per task. The task identity is not given to the model at test time.

In [ ]:
def make_transforms(train=True):
    ops = [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()] if train else []
    ops += [transforms.ToTensor(), transforms.Normalize((0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616))]
    return transforms.Compose(ops)

class ClassIncrementalCIFAR:
    def __init__(self, cfg):
        if cfg.dataset.upper() == 'CIFAR10': Dataset, classes = datasets.CIFAR10, 10
        elif cfg.dataset.upper() == 'CIFAR100': Dataset, classes = datasets.CIFAR100, 100
        else: raise ValueError('This notebook currently provides CIFAR-10/100 loaders; add TinyImageNet here.')
        assert classes % cfg.n_tasks == 0
        self.classes, self.n_tasks = classes, cfg.n_tasks
        self.classes_per_task = classes // cfg.n_tasks
        self.train = Dataset(cfg.data_root, train=True, download=True, transform=make_transforms(True))
        self.test = Dataset(cfg.data_root, train=False, download=True, transform=make_transforms(False))
        self.train_targets = np.asarray(self.train.targets); self.test_targets = np.asarray(self.test.targets)
        self.task_classes = [list(range(t*self.classes_per_task, (t+1)*self.classes_per_task)) for t in range(cfg.n_tasks)]

    def loaders(self, task, batch_size, workers=2):
        cls = self.task_classes[task]
        tr_idx = np.flatnonzero(np.isin(self.train_targets, cls)); te_idx = np.flatnonzero(np.isin(self.test_targets, cls))
        tr = DataLoader(Subset(self.train, tr_idx), batch_size=batch_size, shuffle=True, num_workers=workers, pin_memory=True)
        te = DataLoader(Subset(self.test, te_idx), batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)
        return tr, te

# Uncomment for the real dataset (downloads data):
# stream = ClassIncrementalCIFAR(RUN_CONFIG)

In [ ]:
class ReplayBuffer:
    """Reservoir replay with an optional class-balancing replacement rule."""
    def __init__(self, capacity, device, class_balance=False):
        self.capacity, self.device, self.class_balance = capacity, device, class_balance
        self.examples, self.labels = [], []
        self.seen = 0

    def __len__(self): return len(self.labels)
    def is_empty(self): return len(self) == 0

    def _index(self, label):
        if self.seen < self.capacity: return self.seen
        if self.class_balance and self.labels:
            counts = np.bincount(np.asarray(self.labels), minlength=int(max(self.labels))+1)
            candidates = np.flatnonzero(np.isin(self.labels, np.flatnonzero(counts == counts.max())))
            if np.random.randint(self.seen + 1) < self.capacity: return int(np.random.choice(candidates))
            return -1
        j = np.random.randint(self.seen + 1)
        return int(j) if j < self.capacity else -1

    def add(self, examples, labels):
        for x, y in zip(examples.detach().cpu(), labels.detach().cpu()):
            idx = self._index(int(y)); self.seen += 1
            if idx < 0: continue
            if idx == len(self.examples): self.examples.append(x.clone()); self.labels.append(int(y))
            else: self.examples[idx] = x.clone(); self.labels[idx] = int(y)

    def sample(self, n, transform):
        n = min(n, len(self)); ids = np.random.choice(len(self), n, replace=False)
        x = torch.stack([transform(self.examples[i]) for i in ids]).to(self.device)
        y = torch.tensor([self.labels[i] for i in ids], device=self.device)
        return x, y

In [ ]:
class IdempotentResNet(nn.Module):
    def __init__(self, num_classes, nf=32):
        super().__init__(); self.num_classes = num_classes
        def block(cin, cout, stride):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(),
                                 nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU())
        self.f1 = nn.Sequential(nn.Conv2d(3,nf,3,1,1,bias=False), nn.BatchNorm2d(nf), nn.ReLU(),
                               block(nf,nf,1), block(nf,nf*2,2), block(nf*2,nf*4,2), block(nf*4,nf*8,2),
                               nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.feature_dim = nf*8
        self.label_feature = nn.Sequential(nn.Linear(num_classes, self.feature_dim), nn.LeakyReLU(0.1))
        self.f2 = nn.Linear(self.feature_dim, num_classes)

    def forward(self, x, second_input):
        z = self.f1(x) + self.label_feature(second_input)
        return self.f2(z)

    def first_pass(self, x):
        empty = torch.full((x.size(0), self.num_classes), 1/self.num_classes, device=x.device)
        return self(x, empty)

def one_hot_or_empty(labels, num_classes, p_empty, device):
    use_empty = torch.rand((), device=device) < p_empty
    if use_empty: return torch.full((labels.size(0), num_classes), 1/num_classes, device=device)
    return F.one_hot(labels, num_classes=num_classes).float()

In [ ]:
def ider_step(model, old_model, optimizer, x, y, buffer, cfg, transform):
    model.train(); optimizer.zero_grad(); C = model.num_classes
    y_star = one_hot_or_empty(y, C, cfg.empty_probability, x.device)
    y0 = model(x, y_star); y1 = model(x, y0.softmax(-1))
    loss_ice = 0.5 * (F.cross_entropy(y0, y) + F.cross_entropy(y1, y))
    loss_ide = torch.zeros((), device=x.device); loss_rep = torch.zeros((), device=x.device)

    if old_model is not None and not buffer.is_empty() and cfg.alpha:
        bx, by = buffer.sample(cfg.minibatch_size, transform)
        empty = torch.full((bx.size(0), C), 1/C, device=x.device)
        current_pred = model(bx, empty)
        with torch.no_grad(): stable_pred = old_model(bx, current_pred.softmax(-1))
        loss_ide = F.mse_loss(current_pred, stable_pred)

    if not buffer.is_empty() and cfg.beta:
        bx, by = buffer.sample(cfg.minibatch_size, transform)
        by_star = one_hot_or_empty(by, C, cfg.empty_probability, x.device)
        br0 = model(bx, by_star); br1 = model(bx, br0.softmax(-1))
        loss_rep = F.cross_entropy(br0, by) + F.cross_entropy(br1, by)

    total = loss_ice + cfg.alpha*loss_ide + cfg.beta*loss_rep
    total.backward(); optimizer.step(); buffer.add(x, y)
    return {
        'total': float(total.detach()), 'L_ice': float(loss_ice.detach()),
        'L_ide': float(loss_ide.detach()), 'L_rep-ice': float(loss_rep.detach())
    }

In [ ]:
@torch.no_grad()
def evaluate(model, loaders, device):
    model.eval(); scores = []
    for loader in loaders:
        correct = total = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model.first_pass(x).argmax(1)
            correct += int((pred == y).sum()); total += y.numel()
        scores.append(100*correct/max(total,1))
    return scores

def expected_calibration_error(model, loader, device, bins=15):
    model.eval(); confs=[]; correct=[]
    with torch.no_grad():
        for x,y in loader:
            p = model.first_pass(x.to(device)).softmax(1); c, pred = p.max(1)
            confs.append(c.cpu()); correct.append(pred.cpu().eq(y))
    conf, cor = torch.cat(confs), torch.cat(correct).float(); ece = torch.zeros(())
    for lo, hi in zip(torch.linspace(0,1,bins+1)[:-1], torch.linspace(0,1,bins+1)[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.any(): ece += mask.float().mean() * (cor[mask].mean() - conf[mask].mean()).abs()
    return float(ece*100)

def final_metrics(history):
    final = np.asarray(history[-1], dtype=float); faa = final.mean()
    # Histories are ragged: task j only exists from the moment it is learned.
    peaks = []
    for j, final_score in enumerate(final):
        observed = [row[j] for row in history if len(row) > j]
        peaks.append(max(observed))
    forgetting = np.mean(np.asarray(peaks) - final)
    return {'FAA': float(faa), 'FF': float(forgetting)}

In [ ]:
def run_ider(cfg, stream=None, device=DEVICE):
    seed_everything(0)
    if stream is None: stream = ClassIncrementalCIFAR(cfg)
    model = IdempotentResNet(stream.classes).to(device)
    old_model = None; buffer = ReplayBuffer(cfg.buffer_size, device, cfg.class_balance)
    history, loss_rows, test_loaders = [], [], []
    for task in range(cfg.n_tasks):
        train_loader, test_loader = stream.loaders(task, cfg.batch_size, cfg.num_workers)
        test_loaders.append(test_loader)
        optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1,cfg.epochs*len(train_loader)))
        for epoch in range(cfg.epochs):
            for x,y in tqdm(train_loader, desc=f'task {task+1}/{cfg.n_tasks}, epoch {epoch+1}', leave=False):
                x,y = x.to(device), y.to(device)
                row = ider_step(model, old_model, optimizer, x, y, buffer, cfg, stream.train.transform)
                row.update(task=task, epoch=epoch); loss_rows.append(row); scheduler.step()
        scores = evaluate(model, test_loaders, device); history.append(scores)
        print(f'task {task+1}: FAA over seen tasks = {np.mean(scores):.2f}% | scores = {[round(s,2) for s in scores]}')
        old_model = copy.deepcopy(model).eval()
        for p in old_model.parameters(): p.requires_grad_(False)
    metrics = final_metrics(history)
    metrics['ECE_last_task'] = expected_calibration_error(model, test_loaders[-1], device)
    return model, history, pd.DataFrame(loss_rows), metrics

## 5. Smoke test (fast, no download)

This synthetic test checks the complete forward, replay, checkpoint, and metric paths. It is deliberately tiny and is not a paper result.

In [ ]:
class SyntheticStream:
    def __init__(self, n_tasks=2, classes_per_task=2, n=32):
        self.n_tasks=n_tasks; self.classes_per_task=classes_per_task; self.classes=n_tasks*classes_per_task
        self.train=type('T', (), {'transform': lambda self,x:x})()
        self._loaders=[]
        for t in range(n_tasks):
            ys=torch.tensor(np.tile(np.arange(t*classes_per_task,(t+1)*classes_per_task), n//classes_per_task))
            xs=torch.randn(len(ys),3,32,32)
            ds=torch.utils.data.TensorDataset(xs,ys); self._loaders.append((DataLoader(ds,8,shuffle=True),DataLoader(ds,8)))
    def loaders(self, task, batch_size, workers=0): return self._loaders[task]

smoke_cfg = Config(n_tasks=2, buffer_size=8, epochs=1, batch_size=8, minibatch_size=4, num_workers=0)
_, smoke_history, smoke_losses, smoke_metrics = run_ider(smoke_cfg, SyntheticStream(), device='cpu')
display(smoke_losses.head(), smoke_metrics)

In [ ]:
# Recommended starting run: one seed, one epoch, CIFAR-10.
full_cfg = Config(dataset='CIFAR10', n_tasks=5, buffer_size=200, epochs=1, class_balance=False)
# stream = ClassIncrementalCIFAR(full_cfg)
# model, history, losses, metrics = run_ider(full_cfg, stream, DEVICE)
# print(metrics)
# pd.DataFrame(history, columns=[f'task_{i+1}' for i in range(len(history[-1]))]).plot(marker='o', ylim=(0,100), figsize=(8,4))
# plt.ylabel('Accuracy (%)'); plt.xlabel('Training task'); plt.title('IDER task accuracy'); plt.show()

In [ ]:
def run_ablation(base_cfg, stream=None, device=DEVICE):
    rows=[]
    for name, alpha, beta in [('ER',0.0,1.0), ('ER+ID',1.0,1.0), ('SIM-only',0.0,0.0)]:
        cfg=copy.deepcopy(base_cfg); cfg.alpha=alpha; cfg.beta=beta
        _, _, _, m=run_ider(cfg, stream=stream, device=device)
        rows.append({'method':name, **m})
    return pd.DataFrame(rows)

# Example: ablation_table = run_ablation(full_cfg, stream, DEVICE)
# display(ablation_table)